# Phase 3: Fine-Tuning the Champion Model with RandomizedSearchCV

We have identified our best model: a LightGBM classifier trained on data balanced with SMOTE. This notebook takes the final step of **hyperparameter tuning** to squeeze out the maximum possible performance from this model.

**Goal:** Use `RandomizedSearchCV` to find the optimal combination of parameters for our SMOTE-LGBM pipeline and evaluate the final tuned model.

**Steps:**
1. Load the dataset.
2. Set up a pipeline that combines SMOTE and the LGBMClassifier.
3. Define a search space (a grid of parameters) to explore.
4. Run `RandomizedSearchCV` to find the best parameters.
5. Evaluate the final, tuned model on the test set.

### 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import os

# Key imports for the pipeline and SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Visualization libs used later
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")

### 2. Load and Prepare the Dataset

In [ ]:
DATASET_PATH = 'final_training_dataset.csv' # Assumes you are using the dataset from your 13 PDFs
df = pd.read_csv(DATASET_PATH)

X = df.drop('label', axis=1)
y = df['label']

# Split data into training and testing sets before doing anything else
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("✅ Dataset loaded and split.")

### 3. Create the SMOTE-LGBM Pipeline

We define a two-step pipeline. This ensures that when `RandomizedSearchCV` performs cross-validation, SMOTE is applied only to the training fold at each step, preventing data leakage.

In [ ]:
# Determine the correct k_neighbors for SMOTE
min_samples = y_train.value_counts().min()
k = min_samples - 1 if min_samples > 1 else 1

pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42, k_neighbors=k)),
    ('lgbm', lgb.LGBMClassifier(random_state=42))
])

### 4. Define the Hyperparameter Search Space

This dictionary defines the range of values that `RandomizedSearchCV` will sample from.

In [ ]:
param_dist = {
    'lgbm__n_estimators': [100, 200, 300, 400],
    'lgbm__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'lgbm__num_leaves': [20, 31, 40, 50],
    'lgbm__max_depth': [-1, 10, 20],
    'lgbm__reg_alpha': [0, 0.1, 0.5, 1],
    'lgbm__reg_lambda': [0, 0.1, 0.5, 1]
}

# Note: The double underscore `lgbm__` is how we tell the pipeline which step to apply the parameter to.

### 5. Run Randomized Search Cross-Validation

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import numpy as np

# 1) Inspect NaNs (optional but recommended)
print(X.isna().sum().sort_values(ascending=False).head(20))

# 2) Separate numeric / categorical columns (if you have both)
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(exclude=[np.number]).columns

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ],
    remainder="drop"
)

# 3) Build pipeline: Preprocess -> SMOTE -> LGBM
pipeline = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=42, k_neighbors=2)),
    ("lgbm", lgb.LGBMClassifier(random_state=42))
])

# n_iter controls how many different parameter combinations are tried.
# cv is the number of folds for cross-validation.
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=50, # Increase for a more thorough search, decrease for speed
    cv=3, 
    scoring='f1_weighted',
    n_jobs=-1, # Use all available CPU cores
    random_state=42,
    verbose=1
)

print("Starting hyperparameter tuning...")
random_search.fit(X_train, y_train)
print("✅ Tuning complete.")

### 6. Analyze the Results and Evaluate the Best Model

In [ ]:
print("Best parameters found:")
print(random_search.best_params_)
print("\nBest weighted F1-score during search:")
print(f"{random_search.best_score_:.4f}")

# Use the best found model to make predictions on the test set
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

# --- Final Evaluation ---
print("\n--- Classification Report for Tuned Model on Test Set ---")
label_names = ["BODY_TEXT", "TITLE", "H1", "H2", "H3", "H4"]
print(classification_report(y_test, y_pred, target_names=label_names))

print("--- Confusion Matrix for Tuned Model ---")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix for Tuned SMOTE-LGBM Model')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()